In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats


In [3]:
# Plot styling
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_style("whitegrid")
sns.set_palette("Reds_r")

# Load cleaned data
tb_data = pd.read_csv('kncv_nigeria_tb_data_cleaned.csv')

# Parse dates
date_cols = ['screening_date', 'diagnosis_date', 'treatment_start_date', 'outcome_date']
for col in date_cols:
    tb_data[col] = pd.to_datetime(tb_data[col], errors='coerce')

print("Shape:", tb_data.shape)

Shape: (60000, 27)


In [4]:
# ── TEST 1: TB TYPE vs TREATMENT OUTCOME ─────────────────────────

# Filter confirmed TB patients only
confirmed = tb_data[tb_data['tb_type'].notna() & tb_data['treatment_outcome'].notna()]

# Create contingency table
contingency_table = pd.crosstab(confirmed['tb_type'], confirmed['treatment_outcome'])
print("Contingency Table:")
print(contingency_table)

# Run chi-square test
chi2, p_value, dof, expected = stats.chi2_contingency(contingency_table)

print(f"\nChi-square statistic: {chi2:.4f}")
print(f"P-value: {p_value:.6f}")
print(f"Degrees of freedom: {dof}")

if p_value < 0.05:
    print("\n Result: Significant relationship between TB type and treatment outcome")
else:
    print("\n Result: No significant relationship found")

Contingency Table:
treatment_outcome  Died  Lost to Follow-up  Not Evaluated  Treatment Failed  \
tb_type                                                                       
DR-TB               739                985            223               260   
DS-TB              1841               2173            702               890   
TB/HIV             1215               1528            539               627   

treatment_outcome  Treatment Success  
tb_type                               
DR-TB                           2680  
DS-TB                          15129  
TB/HIV                          6353  

Chi-square statistic: 852.8907
P-value: 0.000000
Degrees of freedom: 8

 Result: Significant relationship between TB type and treatment outcome


In [6]:
# ── TEST 2: PLHIV STATUS vs TREATMENT OUTCOME ────────────────────

# Create contingency table
contingency_plhiv = pd.crosstab(confirmed['is_plhiv'], confirmed['treatment_outcome'])
print("Contingency Table:")
print(contingency_plhiv)

# Run chi-square test
chi2, p_value, dof, expected = stats.chi2_contingency(contingency_plhiv)

print(f"\nChi-square statistic: {chi2:.4f}")
print(f"P-value: {p_value:.6f}")
print(f"Degrees of freedom: {dof}")

if p_value < 0.05:
    print("\n Result: Significant relationship between PLHIV status and treatment outcome")
else:
    print("\n Result: No significant relationship found")

Contingency Table:
treatment_outcome  Died  Lost to Follow-up  Not Evaluated  Treatment Failed  \
is_plhiv                                                                      
No                 2855               3524           1075              1308   
Yes                 940               1162            389               469   

treatment_outcome  Treatment Success  
is_plhiv                              
No                             19453  
Yes                             4709  

Chi-square statistic: 159.9573
P-value: 0.000000
Degrees of freedom: 4

 Result: Significant relationship between PLHIV status and treatment outcome


In [7]:
# ── TEST 3: TREATMENT DURATION ACROSS TB TYPES ───────────────────

# Filter confirmed TB patients with valid duration
duration_data = confirmed[confirmed['treatment_duration_months'].notna()]

# Separate groups
ds_tb = duration_data[duration_data['tb_type'] == 'DS-TB']['treatment_duration_months']
dr_tb = duration_data[duration_data['tb_type'] == 'DR-TB']['treatment_duration_months']
tb_hiv = duration_data[duration_data['tb_type'] == 'TB/HIV']['treatment_duration_months']

print(f"DS-TB median duration: {ds_tb.median():.1f} months")
print(f"DR-TB median duration: {dr_tb.median():.1f} months")
print(f"TB/HIV median duration: {tb_hiv.median():.1f} months")

# Run Kruskal-Wallis test
stat, p_value = stats.kruskal(ds_tb, dr_tb, tb_hiv)

print(f"\nKruskal-Wallis statistic: {stat:.4f}")
print(f"P-value: {p_value:.6f}")

if p_value < 0.05:
    print("\n Result: Significant difference in treatment duration across TB types")
else:
    print("\n Result: No significant difference found")

DS-TB median duration: 6.0 months
DR-TB median duration: 21.0 months
TB/HIV median duration: 6.0 months

Kruskal-Wallis statistic: 12832.9990
P-value: 0.000000

 Result: Significant difference in treatment duration across TB types


## Statistical Analysis Summary

Three statistical tests were conducted to validate patterns observed in the EDA:

**Test 1 — TB Type vs Treatment Outcome (Chi-square):**
A significant relationship was found between TB type and treatment outcome (p < 0.05). DR-TB patients consistently show worse outcomes than DS-TB and TB/HIV patients, confirming that drug resistance is a critical determinant of treatment success.

**Test 2 — PLHIV Status vs Treatment Outcome (Chi-square):**
PLHIV status is significantly associated with treatment outcome (p < 0.05). HIV-positive TB patients have lower success rates, reinforcing the need for integrated TB/HIV care and enhanced support for this population.

**Test 3 — Treatment Duration Across TB Types (Kruskal-Wallis):**
Treatment duration differs significantly across TB types (p < 0.05). DR-TB patients have a median treatment duration of 21 months compared to 6 months for DS-TB and TB/HIV patients — a 3.5x longer treatment journey that contributes to higher dropout rates.